In [ ]:

from collections import Counter
import pandas as pd 
import re
import requests
import json
import time
import itertools
from itertools import islice
import glob
import pprint

import pylab as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import ListedColormap

pp = pprint.PrettyPrinter()
import numpy as np

import matplotlib.pyplot as plt

import regex as re

import os
import sys
import ast
from datetime import datetime



In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import sys

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"
FUNCTIONS_DIR = DATASET_CURATION_CODE_ROOT / "functions"
if not FUNCTIONS_DIR.exists():
    raise FileNotFoundError(
        f"Functions folder not found: {FUNCTIONS_DIR}. "
        "Copy code/dataset_curation/functions to the OneDrive code root or adjust CODE_ROOT."
    )

if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

# Keep legacy string-style path variable because the downstream cells use
# expressions like path + r"processed_data\...".
path = str(DATA_ROOT) + "\\"

from scholarly_lens import lens_scroll_search
from scholarly_lens import expand_term
from scholarly_lens import clean_scholarly_lens
from scholarly_lens import build_search_query_scholarly_lens

#define api key and general search terms
API_KEY = require_env("LENS_API_KEY")


In [ ]:
#open indicator search terms
df_indicator_search_all = pd.read_excel(path+R'manual_data\260105_indicators-list_annotated_via_lens.xlsx', index_col = 0, sheet_name='Indicator to search string')

In [ ]:
#get replacement terms
df_exchangable_terms = pd.read_excel(path+R'manual_data\260105_indicators-list_annotated_via_lens.xlsx', index_col = 0, sheet_name='Exchangable terms dictionary')
replacement_list = df_exchangable_terms.index.to_list()

### 1. abstract, title, fulltext search

In [ ]:
#get fulltext search conditions
df_indicator_search_full = df_indicator_search_all[df_indicator_search_all['Fulltext search'] == 1]

#create indicator - edited search terms dict for interaction with the Lens 
target_indicator_search_full_dict = {}
for target in df_indicator_search_full['Target'].unique(): 
    df_target_sel = df_indicator_search_full[df_indicator_search_full['Target'] == target]
    
    indicator_search_full_dict = {}
    for indicator in df_target_sel['Indicator'].unique(): 
        search_condition_list = df_target_sel['Associated search string'][df_target_sel['Indicator'] == indicator].to_list()
        search_condition_list_rp = []
        for term in search_condition_list:
            expanded = expand_term(term, replacement_list)
            search_condition_list_rp.extend(expanded)
    

        
        indicator_search_full_dict[indicator] = search_condition_list_rp


        

    
    target_indicator_search_full_dict[target] = indicator_search_full_dict
    


target_indicator_search_full_dict

In [ ]:
#export search conditions: 
rows = []

for target, indicators in target_indicator_search_full_dict.items():
    for indicator, conditions in indicators.items():
        for cond in conditions:
            rows.append([target, indicator, cond])

df_search_con_full = pd.DataFrame(rows, columns=["Target", "Indicator", "Search condition"])
df_search_con_full.to_excel(path+R"manual_data\indicator_search_conditions_fulltext.xlsx", index=False)

In [ ]:
set_length_dict_fulltext = {}
for target, indicator in target_indicator_search_full_dict.items():  
    if target == 'general':  #delimit target       
        for indic, search_term_list in indicator.items():
            #if indic in flawed_indicators_un:   
                nr = 1
                for search_term in search_term_list: 
                    print('\n'+'Getting data for: '+search_term)
                    target_nr = target
                    indicator_name = indic
                    search_condition_nr = nr
                    search_terms1 = None
                    search_terms2 = None
                    search_terms3 = None
                    if ";" in search_term: 
                        search_term_split = search_term.split('; ') 
                        search_terms1 = search_term_split[0]
                        search_terms2 = search_term_split[1]
                        if len(search_term_split) == 3: 
                            search_terms3 = search_term_split[2]
                    else:
                        search_terms1 = search_term
                    nr = nr +1
        
        
                    #run the search
                    filename = str(target_nr)+'_'+indicator_name+'_'+str(search_condition_nr)+'_fulltext'
                    date = datetime.today().strftime('%Y%m%d')
                    
                    #build query
                    body = build_search_query_scholarly_lens(search_terms1=search_terms1,    
                                                             search_terms2 = search_terms2,
                                                             search_terms3 = search_terms3,
                                                            size = 500, 
                                                            search_title = True,
                                                            search_abstract = True,
                                                            search_fulltext = True,
                                                            open_access = True, 
                                                            #bundle1_operator="AND",
                                                            year_min = 2000)
                    
                    #get data from the Lens
                    data_list_open = lens_scroll_search(
                        api_key=API_KEY,
                        request_body=body,
                        use_max_scrolls=False,
                        #max_scrolls=5,
                        path = path, 
                        filename = filename, 
                        date = date
                        )
                    
                    #clean and save as csv
                    df_cleaned_open = clean_scholarly_lens(data_list_open, path = path, filename = filename, date = date)
                    set_length_dict_fulltext[filename] = len(df_cleaned_open)
                                
                    #break    
                  
    

### 2. abstract and title search

In [ ]:
df_indicator_search_abtitle = df_indicator_search_all[df_indicator_search_all['Fulltext search'] == 0]

target_indicator_search_abtitle_dict = {}
for target in df_indicator_search_abtitle['Target'].unique(): 
    df_target_sel = df_indicator_search_abtitle[df_indicator_search_abtitle['Target'] == target]
    
    indicator_search_abtitle_dict = {}
    for indicator in df_target_sel['Indicator'].unique(): 
        search_condition_list = df_target_sel.loc[
            df_target_sel['Indicator'] == indicator,
            'Associated search string'
        ].to_list()

        search_condition_list_rp = []
        for term in search_condition_list:
            expanded = expand_term(term, replacement_list)
            search_condition_list_rp.extend(expanded)

        indicator_search_abtitle_dict[indicator] = search_condition_list_rp

    target_indicator_search_abtitle_dict[target] = indicator_search_abtitle_dict

target_indicator_search_abtitle_dict

In [ ]:
missing_cond = ['plants; monitoring',
   'species; monitoring',
   'wildlife; monitoring']

In [ ]:
missing_indicator = []
missing_cond = ['plants; monitoring',
   'species; monitoring',
   'wildlife; monitoring']
for target, indicators in target_indicator_search_abtitle_dict.items():
    if target == 7: 
        for indicator, conditions in indicators.items():
            if indicator != '7.1 Index of coastal eutrophication potential':
                missing_indicator.append(indicator)
                missing_cond = conditions[36:]
                break
missing_indicator
missing_cond


In [ ]:
missing_cond

In [ ]:
#export search conditions: 
rows = []

for target, indicators in target_indicator_search_abtitle_dict.items():
    for indicator, conditions in indicators.items():
        for cond in conditions:
            rows.append([target, indicator, cond])

df_search_con_abtitle = pd.DataFrame(rows, columns=["Target", "Indicator", "Search condition"])
df_search_con_abtitle.to_excel(path+R"manual_data\indicator_search_conditions_abtitle.xlsx", index=False)

In [ ]:
set_length_dict = {}
for target, indicator in target_indicator_search_abtitle_dict.items():  
    if target == 'general': 
        for indic, search_term_list in indicator.items():
            #if indic in missing_indicator: 
                nr = 20
                for search_term in search_term_list:
                    if search_term in missing_cond:
                        print('\n'+'Getting data for: '+search_term)
                        target_nr = target
                        indicator_name = indic
                        search_condition_nr = nr
                        search_terms1 = None
                        search_terms2 = None
                        search_terms3 = None
                        if ";" in search_term: 
                            search_term_split = search_term.split('; ') 
                            search_terms1 = search_term_split[0]
                            search_terms2 = search_term_split[1]
                            if len(search_term_split) == 3: 
                                search_terms3 = search_term_split[2]
                        else:
                            search_terms1 = search_term
                        nr = nr +1
            
            
                        #run the search
                        filename = str(target_nr)+'_'+indicator_name+'_'+str(search_condition_nr)
                        date = datetime.today().strftime('%Y%m%d')
                        
                        #build query
                        body = build_search_query_scholarly_lens(search_terms1=search_terms1,    
                                                                 search_terms2 = search_terms2,
                                                                 search_terms3 = search_terms3,
                                                                size = 500, 
                                                                search_title = True,
                                                                search_abstract = True,
                                                                search_fulltext = False,
                                                                open_access = True, 
                                                                #bundle1_operator="AND",
                                                                year_min = 2000)
                        
                        #get data from the Lens
                        data_list_open = lens_scroll_search(
                            api_key=API_KEY,
                            request_body=body,
                            use_max_scrolls=False,
                            #max_scrolls=5,
                            path = path, 
                            filename = filename, 
                            date = date
                            )
                        
                        #clean and save as csv
                        df_cleaned_open = clean_scholarly_lens(data_list_open, path = path, filename = filename, date = date)
                        set_length_dict[filename] = len(df_cleaned_open)
                                    
                        #break    
                
    

### trash

In [ ]:
#define api key and general search terms 
API_KEY = require_env("LENS_API_KEY")

#define search terms
search_terms1 = [   #general monitoring terms
                    "\"biodiversity[- ]monitoring\"", 
                   "\"ecosystem[- ]monitoring\"",
                   "\"species[- ]monitoring\"",
                    "\"environmental[- ]monitoring\"", 

                    #GBF headline and binary indicators
                   "\"red list of ecosystems\"", 
                   "\"extent of natural ecosystems\"", 
                   "\"area under restoration\"",
                   "\"protected area connectedness index\"",
                   "\"protected area connectedness\"",
                   "\"parc[- ]connectedness\"",
                   "\"coverage of protected areas and other effective area-based conservation measures\"",
                   "\"coverage of protected areas\"",
                   "\"green status of species index\"",
                   "\"proportion of populations within species with an effective population size\"",
                   "\"red list index\"",
                    "\"living planet index\"",
                    "\"proportion of fish stocks within biologically sustainable levels\"",
                    "\"rate of invasive alien species establishment\"",
                    "\"pesticide environment concentration\"",
                    "\"index of coastal eutrophication potential\"",
                    "\"bioclimatic ecosystem resilience index\""


                    #Component and complementary indicators (to be added)
                    #Other indicators from Fabio's paper (to be added)
                  ]

In [ ]:
filename = 'target_1_refined'
date = datetime.today().strftime('%Y%m%d')
API_KEY = require_env("LENS_API_KEY")

search_terms2 = [
    "\"spatial[- ]planning\"",
    "\"spatial[- ]management\"",
    "\"space[- ]planning\"",
    "\"space[- ]management\"",
    "land planning",
    "land management",
    "land managing",
    "sea planning",
    "sea management",
    "sea managing",
    "ocean planning",
    "ocean management",
    "ocean managing",
    "environmental planning",
    "environmental management",
    "environmental managing",
    "\"land[- ]use\"",
    "\"sea[- ]use\"",
    "\"ocean[- ]use\"",
]

#build query
body = build_search_query_scholarly_lens( search_terms1=search_terms1,
                                            #search_terms2=search_terms2, 
                                            size = 1000, 
                                            search_fulltext = True, 
                                            open_access = True, 
                                            year_min = 2010)

#get data from the Lens
data_list_open = lens_scroll_search(
    api_key=API_KEY,
    request_body=body,
    use_max_scrolls=False,
    #max_scrolls=5,
    path = path, 
    filename = filename, 
    date = date
    )

#clean and save as csv
df_cleaned_open = clean_scholarly_lens(data_list_open, path = path, filename = filename, date = date)

In [ ]:
filename = 'target_2_refined'
date = datetime.today().strftime('%Y%m%d')

search_terms2 = [
    "*restor*",
    "\"ecosystem[- ]recovery\"",
    "*rewild*"]


#build query
body = build_search_query_scholarly_lens( search_terms1=search_terms1,
                                            search_terms2=search_terms2, 
                                            size = 1000, 
                                            search_fulltext = True, 
                                            open_access = True, 
                                            year_min = 2010)

#get data from the Lens
data_list_open = lens_scroll_search(
    api_key=API_KEY,
    request_body=body,
    use_max_scrolls=False,
    #max_scrolls=5,
    path = path, 
    filename = filename, 
    date = date
    )

#clean and save as csv
df_cleaned_open = clean_scholarly_lens(data_list_open, path = path, filename = filename, date = date)

In [ ]:
filename = 'target_3_refined'
date = datetime.today().strftime('%Y%m%d')

search_terms3 = [
    "*protect*",
    "*preserv*",
    "*conserve*",
    "*conservatio*",
    
    "\"nature[- ]reserve\"",
    "\"area[- ]based management\"",
    "\"area[- ]based measure\"",
    "\"area[- ]based measures\""
    ]



#build query
body = build_search_query_scholarly_lens( search_terms1=search_terms1,
                                            search_terms2=search_terms3, 
                                            size = 1000, 
                                            search_fulltext = True, 
                                            open_access = True, 
                                            year_min = 2010)

#get data from the Lens
data_list_open = lens_scroll_search(
    api_key=API_KEY,
    request_body=body,
    use_max_scrolls=False,
    #max_scrolls=5,
    path = path, 
    filename = filename, 
    date = date
    )

#clean and save as csv
df_cleaned_open = clean_scholarly_lens(data_list_open, path = path, filename = filename, date = date)

In [ ]:
#### end of script :)

### Trash

In [ ]:
data_list_open = lens_scroll_search(
    api_key=API_KEY,
    request_body=query1,
    use_max_scrolls=True,
    max_scrolls=1,
    path = path, 
    filename = filename, 
    date = date
    )


In [ ]:
# Your query setup
filename = 'target_1'
date = datetime.today().strftime('%Y%m%d')

body = {
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*biodiv*"}},
                            {"query_string": {"query": "title:*ecosystem*"}},
                            {"query_string": {"query": "abstract:*biodiv*"}},
                            {"query_string": {"query": "abstract:*ecosystem*"}}
                        ]
                    }
                },
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:\"spatial[- ]planning\""}},
                            {"query_string": {"query": "title:\"spatial[- ]management\""}},
                            {"query_string": {"query": "title:\"space[- ]planning\""}},
                            {"query_string": {"query": "title:\"space[- ]management\""}},
                            {"query_string": {"query": "title:land planning"}},
                            {"query_string": {"query": "title:land management"}},
                            {"query_string": {"query": "title:land managing"}},
                            {"query_string": {"query": "title:sea planning"}},
                            {"query_string": {"query": "title:sea management"}},
                            {"query_string": {"query": "title:sea managing"}},
                            {"query_string": {"query": "title:ocean planning"}},
                            {"query_string": {"query": "title:ocean management"}},
                            {"query_string": {"query": "title:ocean managing"}},
                            {"query_string": {"query": "title:environmental planning"}},
                            {"query_string": {"query": "title:environmental management"}},
                            {"query_string": {"query": "title:environmental managing"}},
                            {"query_string": {"query": "title:\"land[- ]use\""}},
                            {"query_string": {"query": "title:\"sea[- ]use\""}},
                            {"query_string": {"query": "title:\"ocean[- ]use\""}},
                            
                            {"query_string": {"query": "abstract:\"spatial[- ]planning\""}},
                            {"query_string": {"query": "abstract:\"spatial[- ]management\""}},
                            {"query_string": {"query": "abstract:\"space[- ]planning\""}},
                            {"query_string": {"query": "abstract:\"space[- ]management\""}},
                            {"query_string": {"query": "abstract:land planning"}},
                            {"query_string": {"query": "abstract:land management"}},
                            {"query_string": {"query": "abstract:land managing"}},
                            {"query_string": {"query": "abstract:sea planning"}},
                            {"query_string": {"query": "abstract:sea management"}},
                            {"query_string": {"query": "abstract:sea managing"}},
                            {"query_string": {"query": "abstract:ocean planning"}},
                            {"query_string": {"query": "abstract:ocean management"}},
                            {"query_string": {"query": "abstract:ocean managing"}},
                            {"query_string": {"query": "abstract:environmental planning"}},
                            {"query_string": {"query": "abstract:environmental management"}},
                            {"query_string": {"query": "abstract:environmental managing"}},
                            {"query_string": {"query": "abstract:\"land[- ]use\""}},
                            {"query_string": {"query": "abstract:\"sea[- ]use\""}},
                            {"query_string": {"query": "abstract:\"ocean[- ]use\""}},

                        ]
                    }
                },
                {
                    "range": {
                        "year_published": {
                            "gte": 2010
                        }
                    }
                },

                {
                        "term": {
                            "is_open_access": True
                        }
                    },
                {
                    "terms": {
                        "publication_type": [
                            "journal article",
                            "conference proceedings",
                            "conference proceedings article",
                            "report"
                        ]
                    }
                }
            ]
        }
    },
    #"regex": True,
    "size": 5

    
}

include_fields = [
        "lens_id",
        "title",
        "publication_type",
        "abstract",
        "keywords",
        "source",
        "fields_of_study",
        "year_published",
        "authors",
        "funding",
        "open_access",
        "external_ids",
        "references"
                ]

data_list_open = lens_scroll_search(
    api_key=API_KEY,
    request_body=body,
    include=include_fields,
    use_max_scrolls=True,
    max_scrolls=1,
    path = path, 
    filename = filename, 
    date = date
    )


In [ ]:
#clean and save as csv
df_cleaned_open = clean_scholarly_lens(data_list_open, path = path, filename = filename, date = date)

In [ ]:
#open again: 
import ast

df_cleaned_open = pd.read_csv(path+r'processed_data\20251014_scholarly_total_test_target_1.csv', index_col = 0)

In [ ]:
df_cleaned_open_pdf_link = df_cleaned_open[df_cleaned_open['open_access_pdf_urls'].notna()].drop_duplicates().reset_index(drop = True)

doi_url_dict = {}
for row in df_cleaned_open_pdf_link.index.to_list():
    doi = df_cleaned_open_pdf_link.loc[row]['doi']
    url_list = ast.literal_eval(df_cleaned_open_pdf_link.loc[row]['open_access_pdf_urls'])
    doi_url_dict[doi] = url_list

doi_url_dict

In [ ]:
#edit to keep track of lens_ids

In [ ]:
output_dir = path + r"raw_data\publications_test_open"
report = download_pdfs_by_doi(doi_url_dict, output_dir=output_dir,max_workers=2,
    timeout=30)

In [ ]:
report

In [ ]:
# Convert to DataFrame
df_report = pd.DataFrame(report, columns=["DOI", "Status", "Winning_URL", "Explanation"])
df_report.to_csv(path+r'processed_data\get_pdf_output.csv', index=False)

In [ ]:
doi_map_test = {
    '10.4314/wiojms.v19i1.6': ["https://example.com/thiswillfail.pdf",  "http://fakeurl.org/nothinghere.pdf", "https://www.ajol.info/index.php/wiojms/article/download/195836/189488"],
    '10.1002/eet.1775': ['http://liu.diva-portal.org/smash/get/diva2:1142781/FULLTEXT01', 'https://notarealserver.org/fakefile.pdf'], 
    # A PLOS ONE article (OA). First URL deliberately broken.
    "10.1371/journal.pone.0288329": [
        "https://example.com/thiswillfail.pdf",           # fake fail
        "http://fakeurl.org/nothinghere.pdf"              # fake fail
    ],
    
    # A BMC (BioMed Central) article – also fully OA in Europe PMC.
    "10.1186/s12864-024-10073-y": [
        "https://notarealserver.org/fakefile.pdf"         # fake fail
    ],

    # An eLife paper – OA, Europe PMC always hosts PDF.
    "10.7554/eLife.90583": [
        "https://badurl.org/doesnotexist.pdf"             # fake fail
    ],
}

output_dir = path + r"raw_data\publications_test_open"
report = download_pdfs_by_doi(doi_map_test, output_dir=output_dir,max_workers=2,
    timeout=30)

In [ ]:
report

In [ ]:
#old code

In [ ]:
open_ac_links = list_of_list_of_urls

output_dir = path + r"raw_data\publications_test_open"

report = download_pdfs_for_publications(
    open_ac_links,
    output_dir=output_dir,
    max_workers=2,
    timeout=30,
)


## test set ups

In [ ]:
# Your query setup
filename = 'total_test'
date = datetime.today().strftime('%Y%m%d')

body = {
     "query": {
           "match_phrase":{
                "author.affiliation.name": "Harvard University"
           }
     },
     "size": 1000,
     "sort": [
           {
                "year_published": "desc"
           }
     ]
}

include_fields = ["lens_id", "title", "year_published", "authors"]

data_list = lens_scroll_search(
    api_key=API_KEY,
    request_body=body,
    include=include_fields,
    use_max_scrolls=True,
    max_scrolls=1,
    path = path, 
    filename = filename, 
    date = date
    )


In [ ]:
# Your query setup
filename = 'total_test_ginger_open'
date = datetime.today().strftime('%Y%m%d')

body = {
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "fish"}}
                           
                        ]
                    }
                },
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "klosterneuburg"}}

                        ]
                    }
                },
                {
                    "range": {
                        "year_published": {
                            "gte": 1980
                        }
                    }
                },

                {
                        "term": {
                            "is_open_access": True
                        }
                    },
                {
                    "terms": {
                        "publication_type": [
                            "journal article",
                            "conference proceedings",
                            "conference proceedings article",
                            "report"
                        ]
                    }
                }
            ]
        }
    },
    #"regex": True,
    "size": 100

    
}

include_fields = [
        "lens_id",
        "title",
        "publication_type",
        "abstract",
        "keywords",
        "source",
        "fields_of_study",
        "year_published",
        "authors",
        "funding",
        "open_access",
        "external_ids",
        "references"
                ]

data_list_open = lens_scroll_search(
    api_key=API_KEY,
    request_body=body,
    include=include_fields,
    use_max_scrolls=False,
    #max_scrolls=1,
    path = path, 
    filename = filename, 
    date = date
    )


## target search strings to integrate above

In [ ]:
#### get the data
filename = 'gbf_target_1'

print('Getting scholarly data for: ' + filename)
data_dict = []
url = 'https://api.lens.org/scholarly/search'
include = [
        "lens_id",
        "title",
        "publication_type",
        "abstract",
        "keywords",
        "source",
        "fields_of_study",
        "year_published",
        "authors",
        "funding",
        "source_urls"
                ]
headers = {'Authorization': API_KEY, 'Content-Type': 'application/json'}

# Construct request body
request_body = json.dumps({
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*biodiv*"}},
                            {"query_string": {"query": "title:*ecosystem*"}},
                            {"query_string": {"query": "abstract:*biodiv*"}},
                            {"query_string": {"query": "abstract:*ecosystem*"}}
                        ]
                    }
                },
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:\"spatial[- ]planning\""}},
                            {"query_string": {"query": "title:\"spatial[- ]management\""}},
                            {"query_string": {"query": "title:\"space[- ]planning\""}},
                            {"query_string": {"query": "title:\"space[- ]management\""}},
                            {"query_string": {"query": "title:land planning"}},
                            {"query_string": {"query": "title:land management"}},
                            {"query_string": {"query": "title:land managing"}},
                            {"query_string": {"query": "title:sea planning"}},
                            {"query_string": {"query": "title:sea management"}},
                            {"query_string": {"query": "title:sea managing"}},
                            {"query_string": {"query": "title:ocean planning"}},
                            {"query_string": {"query": "title:ocean management"}},
                            {"query_string": {"query": "title:ocean managing"}},
                            {"query_string": {"query": "title:environmental planning"}},
                            {"query_string": {"query": "title:environmental management"}},
                            {"query_string": {"query": "title:environmental managing"}},
                            {"query_string": {"query": "title:\"land[- ]use\""}},
                            {"query_string": {"query": "title:\"sea[- ]use\""}},
                            {"query_string": {"query": "title:\"ocean[- ]use\""}},
                            
                            {"query_string": {"query": "abstract:\"spatial[- ]planning\""}},
                            {"query_string": {"query": "abstract:\"spatial[- ]management\""}},
                            {"query_string": {"query": "abstract:\"space[- ]planning\""}},
                            {"query_string": {"query": "abstract:\"space[- ]management\""}},
                            {"query_string": {"query": "abstract:land planning"}},
                            {"query_string": {"query": "abstract:land management"}},
                            {"query_string": {"query": "abstract:land managing"}},
                            {"query_string": {"query": "abstract:sea planning"}},
                            {"query_string": {"query": "abstract:sea management"}},
                            {"query_string": {"query": "abstract:sea managing"}},
                            {"query_string": {"query": "abstract:ocean planning"}},
                            {"query_string": {"query": "abstract:ocean management"}},
                            {"query_string": {"query": "abstract:ocean managing"}},
                            {"query_string": {"query": "abstract:environmental planning"}},
                            {"query_string": {"query": "abstract:environmental management"}},
                            {"query_string": {"query": "abstract:environmental managing"}},
                            {"query_string": {"query": "abstract:\"land[- ]use\""}},
                            {"query_string": {"query": "abstract:\"sea[- ]use\""}},
                            {"query_string": {"query": "abstract:\"ocean[- ]use\""}},

                        ]
                    }
                },
                {
                    "range": {
                        "year_published": {
                            "gte": 2010
                        }
                    }
                },
                {
                    "terms": {
                        "publication_type": [
                            "journal article",
                            "conference proceedings",
                            "conference proceedings article",
                            "report"
                        ]
                    }
                }
            ]
        }
    },
    #"regex": True,
    "size": 1000,
    "scroll": "1m",
    "include": include
})


# Start scrolling
scroll_count = 1
scroll(scroll_id=None)

# Save raw data
with open(path + r'raw_data\\'+date+r'_scholarly_' + filename + '.json', 'w') as f:
    json.dump(data_dict, f)


# Process into csv and save to processed data
clean_lens_scholarly(data_dict, filename)

In [ ]:
#### get the data
filename = 'gbf_target_2'
print('Getting scholarly data for: ' + filename)
data_dict = []
url = 'https://api.lens.org/scholarly/search'
include = [
        "lens_id",
        "title",
        "publication_type",
        "abstract",
        "keywords",
        "source",
        "fields_of_study",
        "year_published",
        "authors",
        "funding",
        "source_urls"
                ]
headers = {'Authorization': API_KEY, 'Content-Type': 'application/json'}

# Construct request body
request_body = json.dumps({
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*biodiv*"}},
                            {"query_string": {"query": "title:*ecosystem*"}},
                            {"query_string": {"query": "abstract:*biodiv*"}},
                            {"query_string": {"query": "abstract:*ecosystem*"}}
                        ]
                    }
                },
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*restor*"}},
                            {"query_string": {"query": "title:\"ecosystem[- ]recovery\""}},
                            {"query_string": {"query": "title:*rewild*"}},
                            
                            {"query_string": {"query": "abstract:*restor*"}},
                            {"query_string": {"query": "abstract:\"ecosystem[- ]recovery\""}},
                            {"query_string": {"query": "abstract:*rewild*"}}
                        ]
                    }
                },
                {
                    "range": {
                        "year_published": {
                            "gte": 2010
                        }
                    }
                },
                {
                    "terms": {
                        "publication_type": [
                            "journal article",
                            "conference proceedings",
                            "conference proceedings article",
                            "report"
                        ]
                    }
                }
            ]
        }
    },
    #"regex": True,
    "size": 1000,
    "scroll": "1m",
    "include": include
})


# Save raw data
with open(path + r'raw_data\\'+date+r'_scholarly_' + filename + '.json', 'w') as f:
    json.dump(data_dict, f)


# Process into csv and save to processed data
clean_lens_scholarly(data_dict, filename)

In [ ]:
#### get the data
filename = 'gbf_target_3'
print('Getting scholarly data for: ' + filename)
data_dict = []
url = 'https://api.lens.org/scholarly/search'
include = [
        "lens_id",
        "title",
        "publication_type",
        "abstract",
        "keywords",
        "source",
        "fields_of_study",
        "year_published",
        "authors",
        "funding",
        "source_urls"
                ]
headers = {'Authorization': API_KEY, 'Content-Type': 'application/json'}

# Construct request body
request_body = json.dumps({
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*biodiv*"}},
                            {"query_string": {"query": "title:*ecosystem*"}},
                            {"query_string": {"query": "abstract:*biodiv*"}},
                            {"query_string": {"query": "abstract:*ecosystem*"}}
                        ]
                    }
                },
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*protect*"}},
                            {"query_string": {"query": "title:\"nature[- ]reserve\""}},
                            {"query_string": {"query": "title:*preserv*"}},
                            {"query_string": {"query": "title:*conserve*"}},
                            {"query_string": {"query": "title:*conservatio*"}},
                            {"query_string": {"query": "title:\"area[- ]based management\""}},
                            {"query_string": {"query": "title:\"area[- ]based measure\""}},
                            {"query_string": {"query": "title:\"area[- ]based measures\""}},
                            
                            {"query_string": {"query": "abstract:*protect*"}},
                            {"query_string": {"query": "abstract:\"nature[- ]reserve\""}},
                            {"query_string": {"query": "abstract:*preserv*"}},
                            {"query_string": {"query": "abstract:*conserve*"}},
                            {"query_string": {"query": "abstract:*conservatio*"}},
                            {"query_string": {"query": "abstract:\"area[- ]based management\""}},
                            {"query_string": {"query": "abstract:\"area[- ]based measure\""}},
                            {"query_string": {"query": "abstract:\"area[- ]based measures\""}},

                        ]
                    }
                },
                {
                    "range": {
                        "year_published": {
                            "gte": 2010
                        }
                    }
                },
                {
                    "terms": {
                        "publication_type": [
                            "journal article",
                            "conference proceedings",
                            "conference proceedings article",
                            "report"
                        ]
                    }
                }
            ]
        }
    },
    #"regex": True,
    "size": 1000,
    "scroll": "1m",
    "include": include
})

# Save raw data
with open(path + r'raw_data\\'+date+r'_scholarly_' + filename + '.json', 'w') as f:
    json.dump(data_dict, f)

# Process into csv and save to processed data
clean_lens_scholarly(data_dict, filename)

In [ ]:
#### get the data
filename = 'gbf_target_4'
data_dict = []
url = 'https://api.lens.org/scholarly/search'
include = [
        "lens_id",
        "title",
        "publication_type",
        "abstract",
        "keywords",
        "source",
        "fields_of_study",
        "year_published",
        "authors",
        "funding",
        "source_urls"
                ]
headers = {'Authorization': API_KEY, 'Content-Type': 'application/json'}

# Construct request body
request_body = json.dumps({
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:*biodiv*"}},
                            {"query_string": {"query": "title:*ecosystem*"}},
                            {"query_string": {"query": "abstract:*biodiv*"}},
                            {"query_string": {"query": "abstract:*ecosystem*"}}
                        ]
                    }
                },
                {
                    "bool": {
                        "should": [
                            {"query_string": {"query": "title:\"species[- ]loss\""}},
                            {"query_string": {"query": "title:\"habitat[- ]loss\""}},
                            {"query_string": {"query": "title:(species AND *extinct*)"}},
                            {"query_string": {"query": "title:human wildlife"}},
                            {"query_string": {"query": "title:human wild species"}},
                            {"query_string": {"query": "title:(species AND *threat*)"}},
                            {"query_string": {"query": "title:\"species[- ]recovery\""}},
                            
                            {"query_string": {"query": "abstract:\"species[- ]loss\""}},
                            {"query_string": {"query": "abstract:\"habitat[- ]loss\""}},
                            {"query_string": {"query": "abstract:(species AND *extinct*)"}},
                            {"query_string": {"query": "abstract:human wildlife"}},
                            {"query_string": {"query": "abstract:human wild species"}},
                            {"query_string": {"query": "abstract:(species AND *threat*)"}},
                            {"query_string": {"query": "abstract:\"species[- ]recovery\""}}

                        ]
                    }
                },
                {
                    "range": {
                        "year_published": {
                            "gte": 2010
                        }
                    }
                },
                {
                    "terms": {
                        "publication_type": [
                            "journal article",
                            "conference proceedings",
                            "conference proceedings article",
                            "report"
                        ]
                    }
                }
            ]
        }
    },
    #"regex": True,
    "size": 1000,
    "scroll": "1m",
    "include": include
})

# Save raw data
with open(path + r'raw_data\\'+date+r'_scholarly_' + filename + '.json', 'w') as f:
    json.dump(data_dict, f)

# Process into csv and save to processed data
clean_lens_scholarly(data_dict, filename)